<a href="https://colab.research.google.com/github/TheGenesisAIStory/regulatory-insight-engine/blob/main/fiorellia_eval_adapter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fiorell.IA — Eval Adapter
Valutazione del LoRA adapter `fiorellia_behavior_20260421` su `Qwen2.5-3B-Instruct`.

**Pipeline:**
1. Monta Drive → estrai adapter zip
2. Installa dipendenze
3. Carica base model + adapter PEFT
4. Esegui prompt harness sui 16 casi di eval
5. Confronto baseline vs adapter
6. Scoring automatico per categoria
7. Salva report su Drive + GitHub

> Runtime: **GPU T4** obbligatorio — Runtime → Cambia tipo di runtime → T4 GPU

In [3]:
# — 0. Verifica GPU —
import sys, torch

try:
    import subprocess
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                            "--format=csv,noheader"],
                            capture_output=True, text=True)
    if result.returncode == 0:
        print("GPU OK:", result.stdout.strip())
        USE_GPU = True
    else:
        print("⚠️  nvidia-smi returncode != 0, uso CPU")
        USE_GPU = False
except FileNotFoundError:
    print("⚠️  nvidia-smi non trovato — runtime CPU (modalità fallback)")
    USE_GPU = False

DEVICE = "cuda" if USE_GPU else "cpu"
print(f"Device: {DEVICE} | CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch: {torch.__version__}")

⚠️  nvidia-smi non trovato — runtime CPU (modalità fallback)
Device: cpu | CUDA available: False
PyTorch: 2.10.0+cpu


In [4]:
# ── 1. Configurazione ──────────────────────────────────────────────────────
from pathlib import Path
from datetime import datetime

# ✔ Modifica solo questi due parametri se necessario
ADAPTER_NAME  = "fiorellia_behavior_20260421"
BASE_MODEL    = "Qwen/Qwen2.5-3B-Instruct"

# Percorsi
DRIVE_ZIP       = Path(f"/content/drive/MyDrive/fiorellia-runs/{ADAPTER_NAME}.zip")
ADAPTER_DIR     = Path(f"/content/adapter/{ADAPTER_NAME}")
REPO_RAW        = "https://raw.githubusercontent.com/TheGenesisAIStory/regulatory-insight-engine/main"
EVAL_SET_URL    = f"{REPO_RAW}/fiorellia/eval/eval_set_v0.jsonl"
SYSTEM_PROMPT_URL = f"{REPO_RAW}/fiorellia/prompts/system_prompt.txt"
RUN_TS          = datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_JSON     = Path(f"/content/eval_results_{RUN_TS}.jsonl")
DRIVE_OUTPUT    = Path(f"/content/drive/MyDrive/fiorellia-runs/eval_results_{RUN_TS}.jsonl")

print(f"Adapter : {ADAPTER_NAME}")
print(f"Base model: {BASE_MODEL}")
print(f"Drive zip : {DRIVE_ZIP}")
print(f"Output    : {OUTPUT_JSON}")

Adapter : fiorellia_behavior_20260421
Base model: Qwen/Qwen2.5-3B-Instruct
Drive zip : /content/drive/MyDrive/fiorellia-runs/fiorellia_behavior_20260421.zip
Output    : /content/eval_results_20260423_2251.jsonl


In [ ]:
# ── 2b. Monta Drive + estrai adapter ──────────────────────────────────────
from google.colab import drive
import zipfile, shutil

drive.mount("/content/drive")

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(
        f"Zip non trovato: {DRIVE_ZIP}\n"
        "Assicurati di aver eseguito il notebook di training e che il zip sia su Drive."
    )

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(ADAPTER_DIR.parent)

assert (ADAPTER_DIR / "adapter_config.json").exists(), f"adapter_config.json mancante in {ADAPTER_DIR}"
files = list(ADAPTER_DIR.iterdir())
print(f"Adapter estratto in: {ADAPTER_DIR}")
print(f"File trovati ({len(files)}): {[f.name for f in files]}")

In [5]:
# ── 3. Installa dipendenze ────────────────────────────────────────────────────
%pip install -q transformers>=4.45.0 peft>=0.13.0 accelerate>=1.4.0 bitsandbytes>=0.43.0

print("Verifica versioni:")
import transformers, peft, accelerate, torch
print(f"  transformers : {transformers.__version__}")
print(f"  peft         : {peft.__version__}")
print(f"  torch        : {torch.__version__}")
print(f"  CUDA ok      : {torch.cuda.is_available()}")

Verifica versioni:
  transformers : 5.0.0
  peft         : 0.18.1
  torch        : 2.10.0+cpu
  CUDA ok      : False


In [ ]:
# — 4. Carica tokenizer + modello base —
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"Caricamento tokenizer da {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer OK")

if USE_GPU:
    print("Caricamento base model in 4-bit (GPU)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    print("⚠️  Caricamento base model in float32 su CPU (lento)...")
    model_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True,
    )

model_base.eval()
print(f"Base model OK | device: {next(model_base.parameters()).device}")

In [ ]:
# ── 5. Aggiungi adapter PEFT sul base model ───────────────────────────────────
from peft import PeftModel

print(f"Caricamento adapter da {ADAPTER_DIR}...")
model_adapted = PeftModel.from_pretrained(
    model_base, str(ADAPTER_DIR), is_trainable=False
)
model_adapted.eval()

adapter_params = sum(p.numel() for n, p in model_adapted.named_parameters() if "lora" in n.lower())
print(f"Adapter caricato. LoRA params: {adapter_params/1e6:.2f}M")
print("Pronti: model_base (baseline) e model_adapted (con LoRA)")

In [6]:
# ── 6. Scarica eval_set + system_prompt + funzione generate ──────────────────────
import json, urllib.request

with urllib.request.urlopen(SYSTEM_PROMPT_URL) as r:
    SYSTEM_PROMPT = r.read().decode("utf-8").strip()
print(f"System prompt OK ({len(SYSTEM_PROMPT)} chars)")

with urllib.request.urlopen(EVAL_SET_URL) as r:
    raw = r.read().decode("utf-8")
EVAL_SET = [json.loads(line) for line in raw.strip().splitlines() if line.strip()]
print(f"Eval set OK: {len(EVAL_SET)} casi | {set(e['category'] for e in EVAL_SET)}")

@torch.inference_mode()
def generate(model, user_query: str, max_new_tokens: int = 300) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_query},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(tokens, skip_special_tokens=True).strip()

print("generate() pronta.")

System prompt OK (1315 chars)
Eval set OK: 16 casi | {'unsupported_abstention', 'in_scope_grounded', 'out_of_scope_refusal'}
generate() pronta.


In [ ]:
# ── 7. Prompt Harness: baseline vs adapter su tutti i 16 casi ────────────────────
REFUSAL_KW = ["non posso","fuori perimetro","non rientra","non sono in grado",
              "non fornisco consigli","investment advice","non di mia competenza"]
ABSTAIN_KW = ["non ho fonti","non ho informazioni","non sono in grado di garantire",
              "limitata ai documenti","non posso confermare","non ho recuperato",
              "non dispongo","astensione"]

def auto_score(response: str, category: str) -> str:
    r = response.lower()
    if category == "out_of_scope_refusal":
        return "PASS" if any(k in r for k in REFUSAL_KW) else "FAIL"
    elif category == "unsupported_abstention":
        return "PASS" if any(k in r for k in ABSTAIN_KW) else "PARTIAL"
    elif category == "in_scope_grounded":
        return "PASS" if (len(response) > 80 and not any(k in r for k in REFUSAL_KW)) else "FAIL"
    return "UNKNOWN"

results = []
print(f"Avvio harness su {len(EVAL_SET)} casi...\n")

for i, case in enumerate(EVAL_SET):
    qid = case["id"]; query = case["user_query"]; cat = case["category"]
    print(f"[{i+1:02d}/{len(EVAL_SET)}] {qid} | {cat}")
    resp_b = generate(model_base,    query)
    resp_a = generate(model_adapted, query)
    sc_b   = auto_score(resp_b, cat)
    sc_a   = auto_score(resp_a, cat)
    row = {"id": qid, "category": cat, "query": query,
           "expected": case.get("expected_behavior",""),
           "baseline_response": resp_b, "adapter_response": resp_a,
           "baseline_score": sc_b, "adapter_score": sc_a,
           "improved": (sc_a=="PASS" and sc_b!="PASS"),
           "regressed": (sc_b=="PASS" and sc_a!="PASS")}
    results.append(row)
    flag = " ⬆ IMPROVED" if row["improved"] else (" ⬇ REGRESSED" if row["regressed"] else "")
    print(f"  baseline={sc_b} | adapter={sc_a}{flag}")

print("\nHarness completato!")

In [ ]:
# ── 8. Scoring Summary e Report ───────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(results)

print("=" * 60)
print(" FIORELL.IA EVAL REPORT")
print(f" Run: {RUN_TS} | Adapter: {ADAPTER_NAME}")
print("=" * 60)

for cat in ["in_scope_grounded","unsupported_abstention","out_of_scope_refusal"]:
    sub = df[df["category"]==cat]
    if len(sub) == 0: continue
    b_pass = (sub["baseline_score"]=="PASS").sum()
    a_pass = (sub["adapter_score"]=="PASS").sum()
    n = len(sub)
    delta = a_pass - b_pass
    print(f"\n[{cat}] {n} casi")
    print(f"  baseline PASS: {b_pass}/{n} | adapter PASS: {a_pass}/{n} | delta: {'+' if delta>=0 else ''}{delta}")

total   = len(df)
b_total = (df["baseline_score"]=="PASS").sum()
a_total = (df["adapter_score"]=="PASS").sum()
print("\n" + "=" * 60)
print(f" GLOBALE baseline={b_total}/{total} ({b_total/total*100:.0f}%) | adapter={a_total}/{total} ({a_total/total*100:.0f}%)")
print(f" Improved: {df['improved'].sum()} | Regressed: {df['regressed'].sum()}")
print("=" * 60)

df[["id","category","baseline_score","adapter_score","improved","regressed"]]